In [1]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [2]:
# ------------------------------------------------------------
# connect DuckDB to the integrated PostgreSQL data source
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the DuckDB PostgreSQL extension
# the INSTALL statement is required only the first time the extension is used
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the PostgreSQL database to DuckDB
# the database will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

print("PTS | DuckDB connected to PostgreSQL")

PTS | DuckDB connected to PostgreSQL


In [3]:
# ------------------------------------------------------------
# create a flat view of the integrated data source
# ------------------------------------------------------------

# combine penguin and observation data for analytical exploration
# preserve the original temporal attribute from the source
# derive year as an additional attribute for later analytical decisions
db.sql("""
CREATE OR REPLACE TEMP VIEW penguins_source AS
SELECT
    o.id_penguin,
    o.date_time,
    EXTRACT( YEAR FROM o.date_time )::VARCHAR AS year,
    o.name_study AS study,
    o.sample_number,
    p.name_species AS species,
    o.name_island AS island,
    p.sex,
    o.bill_length_mm,
    o.bill_depth_mm,
    o.flipper_length_mm,
    o.body_mass_g,
    o.latitude,
    o.longitude,
    o.name_batch,
    b.name_lab AS lab
FROM pg.public.observation AS o JOIN pg.public.penguin AS p
     ON p.id = o.id_penguin
     LEFT JOIN pg.public.batch AS b
     ON b.name = o.name_batch;
""")

# check the number of observations represented in the view
db.sql("""
SELECT COUNT(*) AS n_observations
FROM penguins_source;
""").show()

┌────────────────┐
│ n_observations │
│     int64      │
├────────────────┤
│            344 │
└────────────────┘



In [4]:
# ------------------------------------------------------------
# inspect the flat integrated source structure
# ------------------------------------------------------------

# inspect attribute names and data types
schema = db.sql("""
DESCRIBE penguins_source;
""")

schema.project("""
column_name,
column_type
""").show()

# inspect a few observations
db.sql("""
SELECT *
FROM penguins_source
LIMIT 10;
""").show()

┌───────────────────┬─────────────┐
│    column_name    │ column_type │
│      varchar      │   varchar   │
├───────────────────┼─────────────┤
│ id_penguin        │ INTEGER     │
│ date_time         │ TIMESTAMP   │
│ year              │ VARCHAR     │
│ study             │ VARCHAR     │
│ sample_number     │ INTEGER     │
│ species           │ VARCHAR     │
│ island            │ VARCHAR     │
│ sex               │ VARCHAR     │
│ bill_length_mm    │ DOUBLE      │
│ bill_depth_mm     │ DOUBLE      │
│ flipper_length_mm │ INTEGER     │
│ body_mass_g       │ INTEGER     │
│ latitude          │ DOUBLE      │
│ longitude         │ DOUBLE      │
│ name_batch        │ VARCHAR     │
│ lab               │ VARCHAR     │
└───────────────────┴─────────────┘
  16 rows               2 columns

┌────────────┬─────────────────────┬─────────┬─────────┬───────────────┬─────────┬───────────┬─────────┬────────────────┬───────────────┬───────────────────┬─────────────┬──────────┬───────────┬────────────┬──

In [5]:
# ------------------------------------------------------------
# guiding questions
# ------------------------------------------------------------

# IMPORTANT:
# these questions are not answered immediately
# they guide the construction of the following analysis cells
# each following cell should collect evidence relevant to one or more of these questions
# conclusions should be drawn only after the corresponding evidence has been inspected

# GQ1. how is the target species distributed?
# GQ2. is species associated with the island context?
# GQ3. do morphological measurements differ across species?
# GQ4. how does species vary across collection years, and is year analytically relevant?

In [6]:
# ------------------------------------------------------------
# inspect the target distribution
# ------------------------------------------------------------

# count observations for each species
db.sql("""
SELECT
    species,

    -- count the number of observations for each species
    COUNT(*) AS n_observations

FROM penguins_source

-- group observations by species
GROUP BY species

ORDER BY n_observations DESC;
""").show()

# expected result:
# three rows, one for each species
# Adelie should be the most frequent class
# Chinstrap should be the least frequent class

┌───────────┬────────────────┐
│  species  │ n_observations │
│  varchar  │     int64      │
├───────────┼────────────────┤
│ Adelie    │            152 │
│ Gentoo    │            124 │
│ Chinstrap │             68 │
└───────────┴────────────────┘



In [7]:
# ------------------------------------------------------------
# TODO -> DONE
# interpret evidence for GQ1 — target distribution
# ------------------------------------------------------------

# task:
# - use the target counts obtained above
# - answer the evidence question and conclude on GQ1

# E1. which species are most and least represented, and what is the approximate largest-to-smallest ratio?
# A1: A espécie mais representada é 'Adelie' (com 152 observações, aprox. 44.1%) e a menos representada é 'Chinstrap' (com 68 observações).
#     A razão aproximada entre a maior e a menor classe é de 152 / 68 ~= 2,24 : 1 (ou seja, existem cerca de 2,2 vezes mais observações de 'Adelie' do que de 'Chinstrap').

# conclusion for GQ1
# - how is the target species distributed?
# A: A variável-alvo (species) apresenta uma distribuição moderadamente desbalanceada entre as 3 classes ('Adelie': 44,2%, 'Gentoo': 36,0%, 'Chinstrap': 19,8%).
#    Embora exista uma disparidade relativa de ~2,2x entre a classe maioritária e a minoritária, todas as classes possuem representação suficiente na amostragem,
#    não havendo um desequilíbrio extremo que exija reamostragem drástica. No entanto, este desequilíbrio relativo indica que a avaliação de modelos deve utilizar
#    métricas ponderadas em vez de acurácia simples.

In [8]:
# ------------------------------------------------------------
# construct a multidimensional analytical view
# ------------------------------------------------------------

# create the multidimensional aggregation
db.sql("""
CREATE OR REPLACE TEMP VIEW penguins_cube AS
SELECT
    species,
    island,
    sex,
    COUNT(*) AS n_observations,
    ROUND( AVG( bill_length_mm ), 2 ) AS avg_bill_length_mm,
    ROUND( AVG( bill_depth_mm ), 2 ) AS avg_bill_depth_mm,
    ROUND( AVG( flipper_length_mm ), 2 ) AS avg_flipper_length_mm,
    ROUND( AVG( body_mass_g ), 2 ) AS avg_body_mass_g,

    -- identify the aggregation level generated for
    -- species, island and sex using GROUPING_ID
    RIGHT( GROUPING_ID( species, island, sex )::BIT::VARCHAR, 3 ) AS grouping_bits

FROM penguins_source

-- generate all aggregation levels for species, island and sex
GROUP BY CUBE( species, island, sex );
""")

# inspect all aggregation levels generated by the cube
db.sql("""
SELECT *
FROM penguins_cube
ORDER BY grouping_bits, species, island, sex;
""").show()

# count the cells generated at each aggregation level
db.sql("""
SELECT
    grouping_bits,
    COUNT(*) AS n_cells
FROM penguins_cube
GROUP BY grouping_bits
ORDER BY grouping_bits;
""").show()

# expected result:
# 8 aggregation levels, represented by grouping_bits from 000 to 111
# the detailed level (000) contains the largest number of cells
# the grand-total level (111) contains a single cell

┌───────────┬───────────┬─────────┬────────────────┬────────────────────┬───────────────────┬───────────────────────┬─────────────────┬───────────────┐
│  species  │  island   │   sex   │ n_observations │ avg_bill_length_mm │ avg_bill_depth_mm │ avg_flipper_length_mm │ avg_body_mass_g │ grouping_bits │
│  varchar  │  varchar  │ varchar │     int64      │       double       │      double       │        double         │     double      │    varchar    │
├───────────┼───────────┼─────────┼────────────────┼────────────────────┼───────────────────┼───────────────────────┼─────────────────┼───────────────┤
│ Adelie    │ Biscoe    │ FEMALE  │             22 │              37.36 │              17.7 │                187.18 │         3369.32 │ 000           │
│ Adelie    │ Biscoe    │ MALE    │             22 │              40.59 │             19.04 │                190.41 │          4050.0 │ 000           │
│ Adelie    │ Dream     │ FEMALE  │             27 │              36.91 │             17

In [9]:
# ------------------------------------------------------------
# interpret the cube structure
# ------------------------------------------------------------

# the CUBE contains all eight aggregation levels
# induced by three dimensions (species, island, sex)

# the number of cells at each level reflects:
# - the combinations actually present in the data
# - not all theoretically possible combinations

# note that NULL may have two different meanings in a CUBE:
# - a missing value already present in the source data
# - a dimension that has been aggregated at that level
#
# grouping_bits allows these two situations to be distinguished

# in this dataset:
# - species and island have three observed non-null values each
# - sex has two observed non-null values plus missing values
# - therefore, NULL for sex with its grouping bit = 0 represents missing data
# - NULL with the corresponding grouping bit = 1 represents aggregation

#
# questions to investigate:
# 1. why are fewer cells observed than theoretically possible at some aggregation levels?
# 2. which species × island combinations are absent from the data?

In [10]:
# ------------------------------------------------------------
# construct all species × island combinations
# ------------------------------------------------------------

# generate the Cartesian product of the observed species and island values
db.sql("""
CREATE OR REPLACE TEMP VIEW possible_species_island AS
SELECT
    s.species,
    i.island
FROM
    (SELECT DISTINCT species FROM penguins_source) AS s

    -- combine every observed species with every observed island
    CROSS JOIN

    (SELECT DISTINCT island FROM penguins_source) AS i;
""")

db.sql("""
SELECT *
FROM possible_species_island
ORDER BY island, species;
""").show()

# expected result:
# 9 rows corresponding to all combinations
# of 3 observed species × 3 observed islands

┌───────────┬───────────┐
│  species  │  island   │
│  varchar  │  varchar  │
├───────────┼───────────┤
│ Adelie    │ Biscoe    │
│ Chinstrap │ Biscoe    │
│ Gentoo    │ Biscoe    │
│ Adelie    │ Dream     │
│ Chinstrap │ Dream     │
│ Gentoo    │ Dream     │
│ Adelie    │ Torgersen │
│ Chinstrap │ Torgersen │
│ Gentoo    │ Torgersen │
└───────────┴───────────┘



In [11]:
# ------------------------------------------------------------
# identify absent species × island combinations
# ------------------------------------------------------------

# compare all species × island combinations with those observed in the cube
# recall that grouping_bits = '001' selects the level where sex is aggregated

db.sql("""
SELECT
    p.species AS species_absent,
    p.island AS island_absent
FROM possible_species_island AS p

     -- keep all possible combinations and match those observed in the cube
     LEFT JOIN
     penguins_cube AS o

     -- match species and island at the species × island aggregation level
     ON p.species = o.species
     AND p.island = o.island
     AND o.grouping_bits = '001'

-- retain only combinations with no corresponding observed cube cell
WHERE o.species IS NULL

ORDER BY p.species, p.island;
""").show()

# expected result:
# 4 absent species × island combinations
# Adelie should not appear among them

┌────────────────┬───────────────┐
│ species_absent │ island_absent │
│    varchar     │    varchar    │
├────────────────┼───────────────┤
│ Chinstrap      │ Biscoe        │
│ Chinstrap      │ Torgersen     │
│ Gentoo         │ Dream         │
│ Gentoo         │ Torgersen     │
└────────────────┴───────────────┘



In [12]:
# ------------------------------------------------------------
# inspect the observed species × island combinations
# ------------------------------------------------------------

# pivot the species × island level, where sex is aggregated
db.sql("""
PIVOT (
    SELECT
        species,
        island,
        n_observations
    FROM penguins_cube
    WHERE grouping_bits = '001'
)

-- make species values become columns
ON species

-- aggregate the number of observations in each cell
USING SUM( n_observations )

-- keep islands as rows
GROUP BY island

ORDER BY island;
""").show()

#
# in DuckDB PIVOT:
# - ON defines which attribute values become columns
# - GROUP BY defines which attribute values remain as rows
# - USING defines the measure aggregated inside each row × column cell

# expected result:
# 3 rows, one for each island
# some species × island cells should be NULL,
# corresponding to combinations not observed in the data

┌───────────┬────────┬───────────┬────────┐
│  island   │ Adelie │ Chinstrap │ Gentoo │
│  varchar  │ int128 │  int128   │ int128 │
├───────────┼────────┼───────────┼────────┤
│ Biscoe    │     44 │      NULL │    124 │
│ Dream     │     56 │        68 │   NULL │
│ Torgersen │     52 │      NULL │   NULL │
└───────────┴────────┴───────────┴────────┘



In [13]:
# ------------------------------------------------------------
# TODO -> DONE
# interpret evidence for GQ2 — island context
# ------------------------------------------------------------

# task:
# - use the absent combinations and pivot results
# - answer the evidence question and conclude on GQ2

# E1. which species × island combinations are absent?
# A1: As 4 combinações ausentes são: Chinstrap x Biscoe, Chinstrap x Torgersen, Gentoo x Dream e Gentoo x Torgersen.

# conclusion for GQ2
# - what does this pattern indicate about the association between island and species?
# A: Existe uma forte associação geográfica entre a ilha e a espécie na amostragem:
#    A espécie 'Gentoo' ocorre exclusivamente na ilha 'Biscoe' (100% dos registos Gentoo) e a espécie 'Chinstrap' 
#    ocorre exclusivamente na ilha 'Dream' (100% dos registos Chinstrap). Apenas a espécie 'Adelie' é generalista,
#    estando presente em todas as três ilhas (e sendo a única espécie observada na ilha 'Torgersen'). Esta segregação
#    confere à variável 'island' um elevado poder discriminativo contextual para separar a classe 'Adelie' de 'Gentoo'
#    e 'Chinstrap', embora reflita limitações de distribuição territorial da amostragem.

In [14]:
# ------------------------------------------------------------
# inspect morphological measurements across species
# ------------------------------------------------------------

# grouping_bits = '011' keeps species and aggregates island and sex
# AVG ignores NULL values, so each measurement is averaged over its available values

db.sql("""
SELECT
    species,
    avg_bill_length_mm,
    avg_bill_depth_mm,
    avg_flipper_length_mm,
    avg_body_mass_g
FROM penguins_cube
WHERE grouping_bits = '011'
ORDER BY species;
""").show()

┌───────────┬────────────────────┬───────────────────┬───────────────────────┬─────────────────┐
│  species  │ avg_bill_length_mm │ avg_bill_depth_mm │ avg_flipper_length_mm │ avg_body_mass_g │
│  varchar  │       double       │      double       │        double         │     double      │
├───────────┼────────────────────┼───────────────────┼───────────────────────┼─────────────────┤
│ Adelie    │              38.79 │             18.35 │                189.95 │         3700.66 │
│ Chinstrap │              48.83 │             18.42 │                195.82 │         3733.09 │
│ Gentoo    │               47.5 │             14.98 │                217.19 │         5076.02 │
└───────────┴────────────────────┴───────────────────┴───────────────────────┴─────────────────┘



In [15]:
# ------------------------------------------------------------
# TODO -> DONE
# interpret evidence for GQ3 — morphological measurements
# ------------------------------------------------------------

# task:
# - use the average measurements obtained above
# - identify the main species-level differences and conclude on GQ3

# E1. for each measurement, which species has an average that appears most distinct from the other two?
# - bill_length_mm: 'Adelie' (~38,79 mm vs. ~47,5 mm em 'Gentoo' e ~48,83 mm em 'Chinstrap' - bico visivelmente mais curto)
# - bill_depth_mm: 'Gentoo' (~14,98 mm vs. ~18.35 mm em 'Adelie' e ~18,42 mm em 'Chinstrap' - bico significativamente mais fino)
# - flipper_length_mm: 'Gentoo' (~217,19 mm vs. ~189,95 mm em 'Adelie' e ~195,82 mm em 'Chinstrap' - nadadeiras visivelmente mais longas)
# - body_mass_g: 'Gentoo' (~5076,47 g vs. ~3700,95 g em 'Adelie' e ~3733,09 g em 'Chinstrap' - massa corporal substancialmente maior)

# conclusion for GQ3
# - what does this evidence suggest about morphological differences across species?
# A: As medições morfológicas apresentam uma elevada capacidade discriminativa complementar entre as três espécies. A espécie 'Gentoo'
#    distingue-se claramente de 'Adelie' e 'Chinstrap' por ser de maior porte (maior massa corporal e nadadeiras mais longas) e possuir
#    um bico mais fino. Por sua vez, 'Adelie' e 'Chinstrap', embora semelhantes em massa corporal e profundidade de bico, diferem no
#    comprimento do bico (~38,8 mm em 'Adelie' vs. ~48,8 mm em 'Chinstrap'). Esta evidência valida a hipótese de que as quatro características
#    biométricas são preditores numéricos legítimos e fundamentais para a separação das espécies no dataset analítico.

In [16]:
# ------------------------------------------------------------
# inspect species across collection years
# ------------------------------------------------------------

# compare species counts and proportions within each collection year
#
# hint:
# OVER defines a window of rows over which an aggregate can be evaluated
# without collapsing those rows into a single result row
#
# PARTITION BY divides that window into independent groups
# here, the required denominator is:
# - the total number of observations within the corresponding year

db.sql("""
SELECT
    species,
    year,
    COUNT(*) AS n_observations,

    -- calculate the percentage of each species within its collection year
    -- hint: use SUM(COUNT(*)) OVER (...) to obtain the yearly total
    ROUND( 100.0 * COUNT(*) /
           SUM(COUNT(*)) OVER (PARTITION BY year), 2 ) AS percent_within_year

FROM penguins_source
GROUP BY species, year
ORDER BY year, species;
""").show()

# expected result:
# 9 rows: 3 species × 3 collection years
# percentages should sum to approximately 100% within each year

┌───────────┬─────────┬────────────────┬─────────────────────┐
│  species  │  year   │ n_observations │ percent_within_year │
│  varchar  │ varchar │     int64      │       double        │
├───────────┼─────────┼────────────────┼─────────────────────┤
│ Adelie    │ 2007    │             50 │               45.45 │
│ Chinstrap │ 2007    │             26 │               23.64 │
│ Gentoo    │ 2007    │             34 │               30.91 │
│ Adelie    │ 2008    │             50 │               43.86 │
│ Chinstrap │ 2008    │             18 │               15.79 │
│ Gentoo    │ 2008    │             46 │               40.35 │
│ Adelie    │ 2009    │             52 │               43.33 │
│ Chinstrap │ 2009    │             24 │                20.0 │
│ Gentoo    │ 2009    │             44 │               36.67 │
└───────────┴─────────┴────────────────┴─────────────────────┘



In [17]:
# ------------------------------------------------------------
# TODO -> DONE
# interpret evidence for GQ4 — collection year
# ------------------------------------------------------------

# task:
# - use the species counts and within-year percentages obtained above
# - compare the species composition across collection years
# - conclude on GQ4

# E1. which species shows the largest variation in percentage across years?
# A1: A espécie 'Gentoo' apresenta a maior variação percentual ao longo dos anos, oscilando entre 30,91% (em 2007) e 40,45% (em 2008), uma variação de ~9,44
#     pontos percentuais. 'Chinstrap' varia entre 15,79% e 23,64% (~7,85 pp), enquanto a classe 'Adelie' permanece extremamente estável entre 43,33% e 45,45% (~2,12 pp). 

# E2. are the species proportions approximately stable across the three years?
# A2: Sim, as proporções das espécies são aproximadamente estáveis ao longo dos três anos de observação. A hierarquia de prevalência das classes mantém-se rigorosamente
#     idêntica em todos os anos ('Adelie' > 'Gentoo' > 'Chinstrap'), sem alterações drásticas nem roturas estruturais na composição da amostragem.

# conclusion for GQ4
# - what does this evidence suggest about year as a possible analytical attribute?
# A: O ano de recolha ('year') descreve o contexto tempora em que as observações foram registadas (2007, 2008, 2009). Embora a composição das espécies se mantenha 
#    relativamente estável, retém-se a propriedade 'year' como atributo explicativo candidato. A sua presença no dataset analíico permite representar o contexto
#    do estudo e avaliar explicitamente, na fase de modelação posterior, se a dimensão temporal traz informação complementar relevante para a previsão da espécie.

In [18]:
# ------------------------------------------------------------
# guiding questions — attribute roles and provenance
# ------------------------------------------------------------

# IMPORTANT:
# these questions are not answered immediately
# they guide the following cells that collect evidence about source and processing attributes
# conclusions should be drawn only after inspecting that evidence

# prediction-time:
# assume that species is predicted,
# at the end of the field observation and measurement stage,
# i.e., prediction takes place before subsequent sample processing or laboratory assignment

# GQ5. what does sample_number reveal about the data-collection process?
# GQ6. how is batch assignment associated with species?
# GQ7. how is laboratory assignment associated with species, and how does it compare with batch?

In [19]:
# ------------------------------------------------------------
# inspect sample_number across species and studies
# ------------------------------------------------------------

# compare sample_number ranges across study and species

db.sql("""
SELECT
    species,
    study,
    COUNT(*) AS n_observations,

    -- obtain the minimum sample_number in each species × study group
    MIN(sample_number) AS min_sample_number,

    -- obtain the maximum sample_number in each species × study group
    MAX(sample_number) AS max_sample_number

FROM penguins_source

-- define the groups needed to compare sample_number ranges
GROUP BY species, study

ORDER BY species, study;
""").show()

# expected result:
# 9 rows: 3 species × 3 studies
#
# as a consistency check, across the three studies:
# - Adelie:    min = 1, max = 152
# - Chinstrap: min = 1, max = 68
# - Gentoo:    min = 1, max = 124
#
# inspect how the min/max ranges progress from one study to the next
# within each species

┌───────────┬─────────┬────────────────┬───────────────────┬───────────────────┐
│  species  │  study  │ n_observations │ min_sample_number │ max_sample_number │
│  varchar  │ varchar │     int64      │       int32       │       int32       │
├───────────┼─────────┼────────────────┼───────────────────┼───────────────────┤
│ Adelie    │ PAL0708 │             50 │                 1 │                50 │
│ Adelie    │ PAL0809 │             50 │                51 │               100 │
│ Adelie    │ PAL0910 │             52 │               101 │               152 │
│ Chinstrap │ PAL0708 │             26 │                 1 │                26 │
│ Chinstrap │ PAL0809 │             18 │                27 │                44 │
│ Chinstrap │ PAL0910 │             24 │                45 │                68 │
│ Gentoo    │ PAL0708 │             34 │                 1 │                34 │
│ Gentoo    │ PAL0809 │             46 │                35 │                80 │
│ Gentoo    │ PAL0910 │     

In [20]:
# ------------------------------------------------------------
# inspect batch assignment across species
# ------------------------------------------------------------

# compare the distribution of batches across species

db.sql("""
SELECT
    species,
    name_batch,

    -- count the observations for each species × batch combination
    COUNT(*) AS n_observations

FROM penguins_source

-- define the groups needed to compare batch assignment across species
GROUP BY species, name_batch

ORDER BY species, name_batch;
""").show()

# expected result:
# 12 rows
# each row represents one observed species × batch combination
# including combinations where name_batch is NULL

┌───────────┬────────────┬────────────────┐
│  species  │ name_batch │ n_observations │
│  varchar  │  varchar   │     int64      │
├───────────┼────────────┼────────────────┤
│ Adelie    │ B01        │             74 │
│ Adelie    │ B02        │             66 │
│ Adelie    │ B03        │              1 │
│ Adelie    │ NULL       │             11 │
│ Chinstrap │ B04        │              1 │
│ Chinstrap │ B05        │             38 │
│ Chinstrap │ B06        │             25 │
│ Chinstrap │ NULL       │              4 │
│ Gentoo    │ B03        │             54 │
│ Gentoo    │ B04        │             60 │
│ Gentoo    │ B05        │              1 │
│ Gentoo    │ NULL       │              9 │
└───────────┴────────────┴────────────────┘
  12 rows                       3 columns



In [21]:
# ------------------------------------------------------------
# inspect laboratory assignment across species
# ------------------------------------------------------------

# compare the distribution of laboratories across species

# note:
# recall that lab is not a penguin characteristic or an observation measurement
# it reflects subsequent data-processing / organizational context

db.sql("""
SELECT
    species,

    -- include the laboratory attribute
    lab,

    -- count the observations for each species × laboratory combination
    COUNT(*) AS n_observations

FROM penguins_source

-- group observations by species and laboratory
GROUP BY species, lab

ORDER BY species, lab;
""").show()

# expected result:
# 9 rows
# each row represents one observed species × laboratory combination,
# including combinations where lab is NULL

┌───────────┬─────────┬────────────────┐
│  species  │   lab   │ n_observations │
│  varchar  │ varchar │     int64      │
├───────────┼─────────┼────────────────┤
│ Adelie    │ Lab_A   │             75 │
│ Adelie    │ Lab_B   │             66 │
│ Adelie    │ NULL    │             11 │
│ Chinstrap │ Lab_A   │             38 │
│ Chinstrap │ Lab_B   │             26 │
│ Chinstrap │ NULL    │              4 │
│ Gentoo    │ Lab_A   │             55 │
│ Gentoo    │ Lab_B   │             60 │
│ Gentoo    │ NULL    │              9 │
└───────────┴─────────┴────────────────┘



In [22]:
# ------------------------------------------------------------
# TODO -> DONE
# interpret evidence for GQ5–GQ7 — attribute roles and provenance
# ------------------------------------------------------------

# task:
# - interpret the patterns shown by the previous results
# - use those patterns to conclude on GQ5, GQ6 and GQ7

# E1. how do sample_number ranges progress across studies within each species?
# A1: O 'sample_number' começa em 1 no primeiro estudo (PAL0708) de cada espécie e progride de forma contínua e sequencial
#     ao longo dos estudos subsequentes para essa mesma espécie:
#     + 'Adelie': 1-50 -> 51-100 -> 101-152; 
#     + 'Chinstrap': 1-26 -> 27-44 -> 45-68;
#     + 'Gentoo': 1-34 -> 35-80 -> 81-124.

# conclusion for GQ5
# A:  O 'sample_number' é um contador sequencial de registo interno atribuído durante a amostragem de campo para cada espécie.
#     Trata-se de um mero identificador do processo de recolha de dados, sem qualquer significado biológico intrínseco ou valor
#     preditivo causativo. Portanto, deve ser excluído dos atributos preditivos do dataset analítico.

# E2. which species dominates each non-null batch, and what exceptions occur?
# A2: Cada lote (batch) é dominado quase unicamente por uma única espécie:
#     + B01 (74) e B02 (66) são 100% 'Adelie';
#     + B03 é quase exclusivamente 'Gentoo' (54 'Gentoo' vs. 1 'Adelie');
#     + B04 é quase exclusivamente 'Gentoo' (60 'Gentoo' vs. 1 'Chinstrap');
#     + B05 é quase exclusivamente 'Chinstrap' (38 'Chinstrap' vs. 1 'Gentoo);
#     + B06 (25) é 100% 'Chinstrap'. 

# conclusion for GQ6
# A:  Os lotes de processamento foram organizados em laboratório através do agrupamento prévio das amostras por espécie (ocorrendo
#     apenas 3 exceções residuais de 1 amostra, provavelmente devido a trocas/erros pontuais de registo). Como a atribuição do lote
#     é efetuada numa etapa logística posterior à observação de campo e pressupõe o conhecimento prévio da espécie, a utilização de
#     'name_batch' causaria um risco grave de 'data leakage' (alvo/processo). Assim, 'name_batch' deve ser rigorosamente eliminado
#     das variáveis preditivas.

# E3. are all species represented in both laboratories, and how does this compare with the batch pattern?
# A3: Sim, as três espécies estão representadas em proporções significativas e equilibradas em ambos os laboratórios (Lab_A e 
#     Lab_B). Ao contrário do padrão observado nos lotes (que foram fortemente segregados por espécie), os laboratórios processaram
#     amostras de todas as espécies.

# conclusion for GQ7
# A:  Embora a variável 'lab' não apresente segregação direta por espécie (diferindo de 'name_batch'), trata-se de um atributo 
#     administrativo relativo à fase de análise laboratorial. Sendo um evento posterior à observação de campo (fora da fronteira 
#     temporal de previsão no terreno) e sem relação funcional biológica com a espécie, a variável 'lab' deve ser removida dos 
#     atributos preditivos do dataset analítico.

In [23]:
# ------------------------------------------------------------
# TODO -> DONE
# select candidate attributes for the analytical dataset
# ------------------------------------------------------------

# task:
# - classify every attribute in penguins_source as:
#   target, retained candidate explanatory attribute, or excluded attribute
# - give one short justification for every retained or excluded attribute
#
# - base each decision on:
#   evidence collected above,
#   attribute meaning and provenance,
#   and prediction-time availability

# IMPORTANT:
# association with the target is not, by itself, sufficient reason
# to retain an attribute
#
# source or processing attributes may encode information about the target
# without representing legitimate prediction-time information

# target
# - species

# retained candidate explanatory attributes
# - attribute: bill_length_mm
#   justification: Medição biométrica de campo com elevado poder discriminativo
#                  complementar (diferencia Adelie com bicos mais curtos de Chinstrap e Gentoo).
# ===
# - attribute: bill_depth_mm
#   justification: Medição biométrica de campo essencial para distinguir a
#                  espécie Gentoo (bico mais raso/fino) de Adelie e Chinstrap.
# ===
# - attribute: flipper_length_mm
#   justification: Medição biométrica de campo que reflete o tamanho da barbatana, sendo um 
#                  indicador físico fortemente distinto na espécie Gentoo.
# ===
# - attribute: body_mass_g
#   justification: Medição biométrica de campo representativa do porte e massa corporal do espécime.
# ===
# - attribute: island
#   justification: Variável de contexto espacial legítima recolhida no terreno, fornecendo informação
#                  contextual sobre o habitat (Gentoo ocorre exclusivamente em Biscoe e Chinstrap em Dream).
# ===
# - attribute: sex
#   justification: Atributo biométrico e demográfico intrínseco do indivíduo observado no terreno, 
#                  que explica a variação associada ao dimorfismo sexual.
# ===
# - attribute: year 
#   justification: Variável de contexto temporal da amostragem (2007-2009), que permite representar o contexto
#                  do estudo e avaliar explicitamente, na fase de modelação posterior, se a dimensão temporal traz
#                  informação complementar relevante para a previsão da espécie.
# ===

# excluded attributes
# - attribute: id_penguin 
#   justification: Identificador primário do pinguim, sem valor preditivo nem capacidade de generalização
#                  para novos espécimes.
# ===
# - attribute: date_time 
#   justification: Carimbo temporal exato da amostragem, servindo unicamente para identificação da observação.
# === 
# - attribute: study 
#   justification: Identificador do projeto científico de expedição, sem relação funcional com as características 
#                  biológicas do pinguim. 
# === 
# - attribute: sample_number 
#   justification: Contador sequencial de amostragem interna por espécie dentro de cada estudo, tratando-se 
#                  de um mero artefacto do processo de recolha. 
# === 
# - attribute: latitude 
#   justification: Coordenada geográfica pontual, redundante com a localização contextual dada por 'island' e 
#                  sujeita a ruído de amostragem. 
# === 
# - attribute: longitude 
#   justification: Coordenada geográfica pontual, redundante com o contexto regional de habitat definido pela
#                  variável 'island'. 
# === 
# - attribute: name_batch 
#   justification: Atribuição logística de lote efetuada pós-campo no laboratório (amostras agrupadas por espécie);
#                  a sua inclusão causaria data leakage direto. 
# === 
# - attribute: lab 
#   justification: Atributo organizacional do laboratório de análise, fora da fronteira temporal de previsão 
#                   no terreno e sem relação biológica com a espécie.
# ===

# final candidate explanatory attribute set
# - bill_length_mm 
# - bill_depth_mm 
# - flipper_length_mm 
# - body_mass_g 
# - island 
# - sex
# - year